# Benchmark the trained SahayakLLM adapter

Use a Kaggle or Colab runtime with CUDA and internet access. The six configurations reuse the same fixtures and base-model revision. Keep the output directory from completed sessions so the runner can resume.

In [ ]:
import os
from pathlib import Path
import subprocess
import sys

root = Path('/kaggle/working' if Path('/kaggle/working').is_dir() else '/content')
repo = root / 'qwen-finetuning'
if not (repo / '.git').is_dir():
    subprocess.run(['git', 'clone', 'https://github.com/AryanMedigeri08/qwen-finetuning.git', str(repo)], check=True)
os.chdir(repo)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements-gpu.txt'], check=True)

If the Hugging Face adapter is private, attach `C2_adapter.zip` to the notebook and set `adapter_zip` to its attached path. The archive contains only the adapter and tokenizer. You may also use a mounted directory through `SAHAYAK_ADAPTER`.

In [ ]:
import torch

assert torch.cuda.is_available(), 'Select a CUDA accelerator for comparable NF4 measurements.'
print(torch.cuda.get_device_name())
adapter_zip = None  # example: Path('/kaggle/input/your-dataset/C2_adapter.zip')
if adapter_zip:
    subprocess.run([sys.executable, '-m', 'sahayak.provenance', '--adapter', str(adapter_zip)], check=True)

out = root / 'sahayak_benchmark' / 't4'
out.mkdir(parents=True, exist_ok=True)
subprocess.run([sys.executable, '-m', 'sahayak.retrieval', '--evaluate', str(out / 'retrieval.json'),
                '--sweep', str(out / 'e5_retrieval_sweep.csv')], check=True)

In [ ]:
subprocess.run([sys.executable, '-m', 'sahayak.benchmark', '--nf4', '--resume', '--configs',
                'C0', 'C1', 'C3', 'C2', 'C4', 'C5', '--output', str(out)], check=True)
subprocess.run([sys.executable, '-m', 'sahayak.research', '--runs', str(out),
                '--results', str(out / 'tables'), '--figures', str(out / 'figures')], check=True)

## Data budget and independent ratings

Notebook 02 trains the smaller adapters. Point `budget_root` to its `adapters` directory when those weights are available. The full adapter's C4 run above is the full-data reference. Carry out two independent ratings with the rubric in `docs/DEMO.md`. Keep the configuration key separate until the raters finish.

In [ ]:
budget_root = None  # path containing C2_n1000/final, C2_n2000/final, C2_n5000/final, C2_n10000/final
if budget_root:
    for rows in (1_000, 2_000, 5_000, 10_000):
        adapter = Path(budget_root) / f'C2_n{rows}' / 'final'
        assert (adapter / 'adapter_config.json').is_file(), f'Missing budget adapter: {adapter}'
        subprocess.run([sys.executable, '-m', 'sahayak.benchmark', '--nf4', '--resume', '--configs', 'C4',
                        '--adapter', str(adapter), '--output', str(out.parent / f'budget_{rows}')], check=True)

subprocess.run([sys.executable, '-m', 'sahayak.research', '--runs', str(out),
                '--human-sheet', str(out / 'rating_sheet.csv')], check=True)